**Контест 1**

импорты

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split, GridSearchCV

Инфа про датасет

In [ ]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

train.head()

,id,trip_duration_min,distance_km,battery_level_start,temperature_c,wind_speed,demand_index,distance_km_noisy,city_zone,scooter_model,is_weekend,rental_price,avg_price_last_week,route_complexity,driver_experience,weather_rating
0,0,25.599707,7.441135,78.519717,6.128784,8.680982,0.861114,11.102968,center,C,0,16.238355,4.985755,-0.777649,6.163800,3
1,1,57.289287,8.770495,NaN,15.947497,8.812507,0.078338,9.426186,park,A,1,6.231436,3.751414,1.587773,2.568192,3
2,2,45.259667,6.147492,31.714901,26.099837,3.501865,0.072575,6.285575,center,A,0,8.914843,2.666093,2.251650,5.311526,1
3,3,37.926217,7.740660,86.634377,11.560209,NaN,0.850318,8.455020,park,B,1,11.197729,5.835958,0.567339,9.418034,2
4,4,13.581025,3.846835,63.223723,15.542815,7.245579,0.212850,7.641726,suburb,B,0,0.560182,0.058212,1.688476,5.726172,1


In [4]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   1200 non-null   int64  
 1   trip_duration_min    1200 non-null   float64
 2   distance_km          1200 non-null   float64
 3   battery_level_start  1093 non-null   float64
 4   temperature_c        1113 non-null   float64
 5   wind_speed           1099 non-null   float64
 6   demand_index         1200 non-null   float64
 7   distance_km_noisy    1200 non-null   float64
 8   city_zone            1200 non-null   str    
 9   scooter_model        1200 non-null   str    
 10  is_weekend           1200 non-null   int64  
 11  rental_price         1200 non-null   float64
 12  avg_price_last_week  1200 non-null   float64
 13  route_complexity     1085 non-null   float64
 14  driver_experience    1200 non-null   float64
 15  weather_rating       1200 non-null   int64  
dtyp

In [67]:
y = train['rental_price']
X = train.drop(['rental_price', 'id'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


Препроцессинг

In [68]:
categorical_features = ['city_zone', 'scooter_model']

numeric_features = X_train.select_dtypes(exclude=['object']).columns

column_transformer = ColumnTransformer([
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ]), categorical_features),
    
    ('num', SimpleImputer(strategy='median'), numeric_features)
])

column_transformer.set_output(transform='pandas')

X_train_encoded = column_transformer.fit_transform(X_train)
X_test_encoded = column_transformer.transform(X_test)


X_train_encoded

,cat__city_zone_park,cat__city_zone_suburb,cat__scooter_model_B,cat__scooter_model_C,num__trip_duration_min,num__distance_km,num__battery_level_start,num__temperature_c,num__wind_speed,num__demand_index,num__distance_km_noisy,num__is_weekend,num__avg_price_last_week,num__route_complexity,num__driver_experience,num__weather_rating
331,0.0,0.0,0.0,0.0,46.681536,12.878301,78.417737,19.413625,1.716782,0.856730,13.417822,1.0,8.395382,0.611676,3.837268,5.0
409,0.0,0.0,0.0,1.0,10.020767,2.816412,58.807769,17.300043,3.027676,0.803133,4.266623,0.0,-0.580652,0.554575,5.515325,4.0
76,0.0,0.0,0.0,1.0,47.419869,8.955663,97.904033,15.637641,6.640122,0.367170,9.341722,0.0,9.842615,0.579244,2.618505,1.0
868,0.0,1.0,0.0,0.0,52.398209,8.321015,80.365549,15.161568,6.292192,0.233396,7.950011,0.0,3.968092,0.433548,9.069104,1.0
138,0.0,1.0,1.0,0.0,24.999628,4.525668,26.498371,22.527440,8.816353,0.325581,4.110334,0.0,4.538845,1.409062,6.753879,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1044,0.0,1.0,0.0,0.0,53.728776,16.895001,80.973967,20.935062,6.542358,0.470762,17.771787,0.0,6.405431,0.607694,7.712572,2.0
1095,0.0,0.0,0.0,0.0,14.709042,2.194585,81.579730,17.162069,4.238082,0.670747,1.767647,0.0,2.310406,0.152188,6.600301,4.0
1130,0.0,0.0,0.0,0.0,44.788106,9.268782,94.413216,7.645126,4.691775,0.020402,9.388655,0.0,7.066071,1.125759,7.370844,4.0
860,0.0,1.0,0.0,1.0,47.654019,7.574047,88.915655,6.121514,2.703486,0.197314,6.431406,1.0,7.429826,0.554575,0.585976,3.0


**Линейная модель**

In [69]:
linear_model = LinearRegression()
linear_model.fit(X_train_encoded, y_train)

linear_y_pred = linear_model.predict(X=X_test_encoded)

ошибка модели


In [70]:
r2 = r2_score(y_test, linear_y_pred)
r2

0.8123655530184463

**Random forest**

In [ ]:
rf_model = RandomForestRegressor(n_estimators=200, max_depth=50, n_jobs=-1)

param_grid = {
    'n_estimators': [200, 300, 500, 700],
    'max_depth': [5, 8, 10, 12]
}

grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=5, scoring='r2')
grid_search.fit(X_train_encoded, y_train)

print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучший R^2 на кросс-валидации: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_encoded)

Лучшие параметры: {'max_depth': 8, 'n_estimators': 500}
Лучший R^2 на кросс-валидации: 0.8257


Ошибка модели

In [ ]:
# r2 = r2_score(y_test, rf_y_pred)
# r2

0.8337427014528693